# c09：方案 C 嵌入断点切分（交互）

按 [v1.1.7 §2](../doc/plan/v1.1.7-semantic-chunking-bcd-plan.md) 与 [`chunking/breakpoint_embed.py`](../src/chunking/breakpoint_embed.py)，先对全文 **只 embed 一次** 得到相邻窗余弦；再列出 **断点阶段（未做 min 合并）** 的小段、查看 **余弦分位数**，**手动设 `USER_SIM_THRESHOLD`** 后查看 **min 合并 + max 劈分** 的最终块。

**规则**：相邻窗余弦 **&lt; `USER_SIM_THRESHOLD`** 则在窗边界记断点；再 `_merge_ranges_min_len`、再 `_split_range_max_len`。

**依赖**：`uv sync --extra embedding`；项目根 `.env`。在 `notebooks/` 下运行首格会自动定位仓库根。

**与** [`scripts/c03_breakpoint_export.py`](../scripts/c03_breakpoint_export.py) **关系**：同一套区间逻辑；导出脚本仍按文件重算 embed。本笔记本用 `compute_adjacent_window_cosines` 缓存，避免为试阈值反复编码。


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)

from dotenv import load_dotenv

load_dotenv(_ROOT / ".env", override=False)
print("仓库根:", _ROOT)


仓库根: /Users/zheng/projects/python/ai/rag/rag_law


## 1. 输入：目录 **或** 文件列表

- **二选一**：设 `INPUT_DIR` 扫描目录内全部 `*.md`；或设 `INPUT_FILES` 指定文件。
- **仅文件名**（如 ``婚姻法.md`` / ``md3__婚姻法.md``）：在 `SEARCH_DIR` 下解析（默认 `data/md_minerU`）。
- **相对项目根路径**（如 ``data/md_minerU/md3__婚姻法.md``）：按路径打开。

下方可调断点参数（与 `c03_breakpoint_export.py` 一致）。


In [2]:
# --- 按需修改 ---
# 模式 A：扫描目录（INPUT_FILES 为 None 时生效）
INPUT_DIR: str | None = "data/md_minerU"
LIMIT: int | None = None  # 仅目录模式：最多前 N 个（字典序）

# 模式 B：指定文件（非 None 时忽略 INPUT_DIR 扫描，仍用 SEARCH_DIR 解析纯文件名）
# 示例：只切「婚姻法」——可写 basename 或完整相对路径
INPUT_FILES: list[str] | None = [
    "婚姻法.md",
]
# 若上面为 None，则走 INPUT_DIR；若给列表，则只处理这些文件

# 纯文件名所在目录（INPUT_FILES 里无 ``/`` 时使用）
SEARCH_DIR = "data/md_minerU"

# 断点参数（与 scripts/c03_breakpoint_export.py 默认一致）
WINDOW_CHARS = 256
STRIDE_CHARS = 128
USER_SIM_THRESHOLD = 0.72  # 手动调参：相邻余弦 < 此值则断点（看完分位数后改）
SIM_THRESHOLD = USER_SIM_THRESHOLD  # 与 c03 参数名对齐
MIN_CHARS = 200
MAX_CHARS = 2200
SPLIT_OVERLAP = 0

# 可选：写入 data/chunk_md/c03_breakpoint（最后一节）
WRITE_EXPORT = False


## 2. 配置与 Embedder

In [3]:
from conf.settings import get_settings
from embeddings import build_embedder

get_settings.cache_clear()
settings = get_settings()

embedder = build_embedder(settings)
print(
    "BGE 维度:",
    embedder.dense_dimension,
    "| CHUNK 相关（入库）:",
    settings.chunk_size,
    settings.chunk_overlap,
)


/Users/zheng/projects/python/ai/rag/rag_law/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 220.12it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 20.33it/s]

BGE 维度: 1024 | CHUNK 相关（入库）: 1500 100


## 3. 解析待处理的 Markdown 路径

In [4]:
def resolve_md_paths(
    root: Path,
    *,
    input_dir: str | None,
    input_files: list[str] | None,
    search_dir: str,
    limit: int | None,
) -> list[Path]:
    search_base = (root / search_dir).resolve()

    if input_files:
        out: list[Path] = []
        for raw in input_files:
            p = Path(raw)
            if p.is_absolute():
                rp = p.resolve()
            else:
                s = str(raw).replace("\\", "/")
                if "/" in s:
                    rp = (root / p).resolve()
                else:
                    rp = (search_base / p.name).resolve()
                    if not rp.is_file():
                        # 短文件名（如 婚姻法.md）时尝试 *婚姻法*.md 唯一匹配（如 md3__婚姻法.md）
                        matches = sorted(search_base.glob("*%s*.md" % p.stem))
                        if len(matches) == 1:
                            rp = matches[0].resolve()
                        elif len(matches) > 1:
                            raise FileNotFoundError(
                                "SEARCH_DIR 下有多处匹配 %r: %s"
                                % (p.name, [m.name for m in matches[:12]])
                            )
            if not rp.is_file():
                raise FileNotFoundError("不是文件或不存在: %s" % rp)
            if rp.suffix.lower() != ".md":
                raise ValueError("仅支持 .md: %s" % rp)
            out.append(rp)
        return sorted(set(out))

    base = (root / input_dir).resolve() if input_dir else search_base
    if not base.is_dir():
        raise FileNotFoundError(base)
    paths = sorted(base.glob("*.md"))
    if limit is not None:
        paths = paths[: max(0, limit)]
    return paths


md_paths = resolve_md_paths(
    _ROOT,
    input_dir=INPUT_DIR,
    input_files=INPUT_FILES,
    search_dir=SEARCH_DIR,
    limit=LIMIT,
)
print("待处理 %s 个文件:" % len(md_paths))
for p in md_paths:
    print(" ", p.relative_to(_ROOT))


待处理 1 个文件:
  data/md_minerU/md3__婚姻法.md


## 4. 一次编码 → 原始断点小段 → 余弦分布 → 手动阈值 → 最终块

### 4.1 缓存相邻窗余弦（embed 一次）

1. **§4.1** 每个文件 `compute_adjacent_window_cosines`（只跑 **一次** BGE）。
2. **§4.2** 列出 **断点阶段** 的全部小段（`min_chars` 合并**之前**；阈值用 `LIST_RAW_SEGMENTS_THRESHOLD`，默认等于 `USER_SIM_THRESHOLD`）。
3. **§4.3** 打印相邻余弦 **min / max / mean** 与 **p10、p25、p50、p75、p90** 等；可选 **直方图**。
4. **§4.4** 修改 **`USER_SIM_THRESHOLD`**，再跑 **§4.5**：诊断 + 最终切块 HTML。


In [5]:
from chunking.breakpoint_embed import (
    adjacent_cosine_percentiles,
    breakpoint_embed_diagnostics_from_pack,
    breakpoint_pipeline_stages,
    compute_adjacent_window_cosines,
    raw_breakpoint_ranges,
)

TEXTS: dict[str, str] = {}
PACKS: dict[str, dict] = {}

for md in md_paths:
    text = md.read_text(encoding="utf-8")
    try:
        sp = md.resolve().relative_to(_ROOT).as_posix()
    except ValueError:
        sp = md.name
    TEXTS[sp] = text
    PACKS[sp] = compute_adjacent_window_cosines(
        text,
        embedder,
        window_chars=WINDOW_CHARS,
        stride_chars=STRIDE_CHARS,
    )
    ac = len(PACKS[sp]["adjacent_cosine"])
    print("已缓存:", sp, "| 全文", len(text), "字 | 相邻余弦对数", ac)


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]

已缓存: data/md_minerU/md3__婚姻法.md | 全文 4619 字 | 相邻余弦对数 35


### 4.2 断点阶段「全部小段」（未做 `min_chars` 合并）

列出 `raw_breakpoint_ranges`：仅在 **余弦 &lt; `LIST_RAW_SEGMENTS_THRESHOLD`** 处切开。默认与 `USER_SIM_THRESHOLD` 相同；若想先看更碎的断点，可把 `LIST_RAW_SEGMENTS_THRESHOLD` **调高**（例如 0.88），再运行本格。


In [6]:
import html
from IPython.display import HTML, display

LIST_RAW_SEGMENTS_THRESHOLD = USER_SIM_THRESHOLD
PREVIEW_HEAD = 240  # 每段只展示前 N 字；None = 全文


def show_raw_segments_for_pack(text: str, pack: dict, sp: str, th: float) -> None:
    n = pack["text_len"]
    raw = raw_breakpoint_ranges(n, pack["starts"], pack["adjacent_cosine"], th)
    display(
        HTML(
            "<h4 style='font-family:system-ui;'>%s</h4><p style='color:#64748b;'>"
            "断点阶段小段数（未 min 合并）: <b>%s</b> | LIST_RAW_SEGMENTS_THRESHOLD=%s</p>"
            % (html.escape(sp), len(raw), th)
        )
    )
    for i, (a, b) in enumerate(raw):
        piece = text[a:b]
        shown = piece if PREVIEW_HEAD is None else piece[:PREVIEW_HEAD]
        tail = "" if PREVIEW_HEAD is None or len(piece) <= PREVIEW_HEAD else " …（省略）"
        meta = "raw #%s | char [%s, %s) | len=%s" % (i, a, b, b - a)
        display(
            HTML(
                "<div style='border:1px solid #94a3b8;border-radius:6px;margin:6px 0;padding:10px;"
                "background:#f8fafc;font-family:ui-monospace,monospace;font-size:12px;'>"
                "<div style='color:#0369a1;font-weight:600;margin-bottom:6px;'>%s</div>"
                "<pre style='white-space:pre-wrap;word-break:break-word;margin:0;color:#0f172a;'>%s%s</pre></div>"
                % (html.escape(meta), html.escape(shown), tail)
            )
        )


for sp in sorted(PACKS.keys()):
    show_raw_segments_for_pack(TEXTS[sp], PACKS[sp], sp, LIST_RAW_SEGMENTS_THRESHOLD)


### 4.3 相邻余弦分布（分位数 + 可选直方图）


In [7]:
import math

for sp in sorted(PACKS.keys()):
    adj = PACKS[sp]["adjacent_cosine"]
    pct = adjacent_cosine_percentiles(adj)
    print("===", sp, "===")
    if not pct:
        print("  单窗或无相邻对，无分布。")
        continue
    order = ["p5", "p10", "p25", "p50", "p75", "p90", "p95"]
    keys = [k for k in order if k in pct]
    line = "  " + " | ".join("%s=%.4f" % (k, pct[k]) for k in keys)
    print(line)
    print(
        "  min=%.4f max=%.4f mean=%.4f count=%d"
        % (pct["min"], pct["max"], pct["mean"], int(pct["count"]))
    )

try:
    import matplotlib.pyplot as plt

    n_files = len(PACKS)
    if n_files == 0:
        print("（PACKS 为空，跳过直方图）")
    else:
        ncols = min(2, max(1, n_files))
        nrows = int(math.ceil(n_files / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows))
        if n_files == 1:
            axes = [axes]
        else:
            axes = axes.flatten().tolist()
        for ax, sp in zip(axes, sorted(PACKS.keys())):
            adj = PACKS[sp]["adjacent_cosine"]
            if not adj:
                ax.set_title(sp[:40] + "…" if len(sp) > 40 else sp)
                ax.text(0.5, 0.5, "无相邻余弦", ha="center")
                continue
            ax.hist(adj, bins=32, color="#38bdf8", edgecolor="#0f172a", alpha=0.85)
            ax.axvline(USER_SIM_THRESHOLD, color="#ef4444", linestyle="--", label="USER")
            for pk in (25, 50, 75):
                kk = "p%d" % pk
                p = adjacent_cosine_percentiles(adj)[kk]
                ax.axvline(p, color="#94a3b8", linestyle=":", linewidth=1)
            ax.set_title(sp[:48] + ("…" if len(sp) > 48 else ""), fontsize=9)
            ax.set_xlabel("adjacent cosine")
            ax.legend(fontsize=7)
        for j in range(n_files, len(axes)):
            fig.delaxes(axes[j])
        plt.tight_layout()
        plt.show()
except ImportError:
    print("（未安装 matplotlib，已跳过直方图）")


=== data/md_minerU/md3__婚姻法.md ===
  p5=0.8166 | p10=0.8375 | p25=0.8665 | p50=0.8910 | p75=0.9116 | p90=0.9258 | p95=0.9301
  min=0.7308 max=0.9719 mean=0.8849 count=35
（未安装 matplotlib，已跳过直方图）


### 4.4 手动指定阈值

修改 **`USER_SIM_THRESHOLD`**（余弦 **低于** 该值则断点）。可对照 **4.3** 的 p25/p50/p75。改完后只须重跑 **§4.5**（不必重跑 §4.1，除非改了 `WINDOW_CHARS` / `STRIDE_CHARS`）。


In [8]:
USER_SIM_THRESHOLD = 0.9
SIM_THRESHOLD = USER_SIM_THRESHOLD


### 4.5 诊断 + 最终切块（基于缓存，不重复 embed）


In [10]:
import html
from IPython.display import HTML, display


def print_diagnostics_from_pack(pack: dict, label: str) -> None:
    d = breakpoint_embed_diagnostics_from_pack(
        pack,
        sim_threshold=USER_SIM_THRESHOLD,
        min_chars=MIN_CHARS,
        max_chars=MAX_CHARS,
    )
    ac = d["adjacent_cosine"]
    n_ac = len(ac)
    head = " | ".join("%.3f" % x for x in ac[:6])
    tail = " | ".join("%.3f" % x for x in ac[-6:]) if n_ac > 12 else ""
    lines = [
        "<b>%s</b> 诊断（USER_SIM_THRESHOLD=%s）：" % (html.escape(label), USER_SIM_THRESHOLD),
        "全文 %s 字 | 滑动窗数 %s | 相邻余弦对数 %s"
        % (d["text_chars"], d["window_count"], n_ac),
        "余弦 min=%s mean=%s | 低于阈值的边数: <b>%s</b>"
        % (
            "—" if d["cosine_min"] is None else "%.3f" % d["cosine_min"],
            "—" if d["cosine_mean"] is None else "%.3f" % d["cosine_mean"],
            d["edges_below_threshold"],
        ),
        "相似度阶段区间数(合并前/后): %s / %s"
        % (d["ranges_before_merge"], d["merged_range_count"]),
        "是否<strong>退化为仅靠 MAX_CHARS 硬切</strong>: <b>%s</b>"
        % ("是" if d["only_max_chars_split"] else "否"),
        "预估最终块数: %s" % d["max_chars_slices_est"],
    ]
    if n_ac:
        lines.append("相邻余弦前 6: " + head)
        if tail:
            lines.append("… 后 6: " + tail)
    display(
        HTML(
            "<div style='font-family:system-ui;font-size:13px;line-height:1.5;color:#334155;"
            "border-left:4px solid #f59e0b;padding:8px 12px;margin:8px 0;background:#fffbeb;'>"
            + "<br/>".join(lines)
            + "</div>"
        )
    )


def show_final_chunks_from_pack(
    text: str, pack: dict, *, source_file: str, source_path: str
) -> list:
    st = breakpoint_pipeline_stages(
        text,
        pack,
        sim_threshold=USER_SIM_THRESHOLD,
        min_chars=MIN_CHARS,
        max_chars=MAX_CHARS,
        split_overlap=SPLIT_OVERLAP,
    )
    final = st["final_ranges"]
    display(
        HTML(
            "<h3 style='font-family:system-ui;'>%s</h3><p style='color:#64748b;'>"
            "最终块数: <b>%s</b></p>"
            % (html.escape(source_path), len(final))
        )
    )
    out = []
    for idx, (cs, ce) in enumerate(final):
        chunk_text = text[cs:ce]
        meta = "chunk_index=%s | char [%s, %s) | len=%s" % (idx, cs, ce, len(chunk_text))
        body = html.escape(chunk_text)
        display(
            HTML(
                "<div style='border:1px solid #475569;border-radius:8px;margin:10px 0;padding:12px;"
                "background:#0f172a;color:#e2e8f0;font-family:ui-monospace,monospace;font-size:13px;'>"
                "<div style='color:#38bdf8;margin-bottom:8px;'>%s</div>"
                "<pre style='white-space:pre-wrap;word-break:break-word;margin:0;'>%s</pre></div>"
                % (html.escape(meta), body)
            )
        )
        out.append((idx, cs, ce, chunk_text))
    return out


all_results: dict[str, list] = {}
for sp in sorted(PACKS.keys()):
    text = TEXTS[sp]
    pack = PACKS[sp]
    print_diagnostics_from_pack(pack, sp)
    md_name = sp.split("/")[-1]
    ch = show_final_chunks_from_pack(text, pack, source_file=md_name, source_path=sp)
    all_results[sp] = ch
print("完成。all_results[source_path] = [(idx, char_start, char_end, text), ...]")


完成。all_results[source_path] = [(idx, char_start, char_end, text), ...]


## 5.（可选）写入 `data/chunk_md/c03_breakpoint/`

将上面同一参数（`USER_SIM_THRESHOLD` 等）导出为带醒目分隔条的 `.md`（与 `c03_breakpoint_export.py` 相同）。设上面 `WRITE_EXPORT = True` 后运行本节。


In [12]:
from chunking.breakpoint_embed import export_breakpoint_chunks_dir

WRITE_EXPORT=1
if WRITE_EXPORT:
    written = export_breakpoint_chunks_dir(
        md_paths,
        _ROOT / "data" / "chunk_md" / "c03_breakpoint",
        embedder,
        window_chars=WINDOW_CHARS,
        stride_chars=STRIDE_CHARS,
        sim_threshold=USER_SIM_THRESHOLD,
        min_chars=MIN_CHARS,
        max_chars=MAX_CHARS,
        split_overlap=SPLIT_OVERLAP,
        progress=True,
    )
    for w in written:
        print("写入:", w.relative_to(_ROOT))
else:
    print("已跳过（WRITE_EXPORT=False）")


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 173.15it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:03<00:00,  3.29s/it]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 412.05it/s]

c03 breakpoint: 100%|██████████| 1/1 [00:04<00:00,  4.57s/file, md3__婚姻法.md]

写入: data/chunk_md/c03_breakpoint/md3__婚姻法.md
